# Essentials for Data Scientists 2024/25 - Group Assignment

The group assignment has been designed for groups of 6 students.  
Groups of 5 or 4 students are also possible.  
The groups have already been formed in Brightspace.

In Brightspace you will also find a separate `data.zip` file with the dataset.

## Learning goals

You will apply the Python, SQL and Git skills to a raw real-world dataset.  
In particular, you will practice:

- (Python language):
    - Processing multiple files (e.g. all files in a directory).
    - Reading different file formats (e.g. TSV, CSV; possibly gzipped).
    - Getting data from a remote server (JSON format).
    - Handling time/date columns (epoch time, time zones).
    - Providing a class interface to the database.
    - Signalling errors to the user.
    - Writing command line tools (with the [Click](https://click.palletsprojects.com/en/7.x/) library).
    - Documenting the code.
- (Python data science libraries):
    - Working with raw (not cleaned) datasets.
    - Understanding the structure of the dataset.
    - Processing the dataset with the [pandas](https://pandas.pydata.org/) library.
    - Visualizing the data with the [matplotlib](https://matplotlib.org/) or [seaborn](https://seaborn.pydata.org/) library.
    - Applying simple machine learning methods (e.g. regression, clustering, etc.) to the dataset.
- (SQL):
    - Creating a relational database (through [SQLAlchemy](https://www.sqlalchemy.org/) library).
    - Inserting data into the database.
    - Querying the database.
- (Git/GitHub):
    - Dividing the work into small tasks and assigning them to team members.
    - Using Git/GitHub for collaborative development of the project.
    - Providing an overview of the project in the `README.md` file.

## Project context

To achieve better energy efficiency, people want to measure and control the energy usage of devices installed in their homes. Many recent home automation devices provide remote access to their state. For example, a smart wall light socket ([example](https://www.fibaro.com/en/products/switches/)) can be remotely:

- controlled (commands: *turn on*, *turn off*, *toggle*, always turn off 5 minutes after turning on, etc.),
- monitored (is the state of the socket *on* or *off*, after how many minutes will it turn off, etc.).

## Data sources

A uniform communication standard with home automation devices is emerging ([Matter](https://en.wikipedia.org/wiki/Matter_(standard))) but not yet commonly available. Consequently, in practice, data from different devices needs to be carefully filtered, adjusted and integrated.

The data in this assignment originates from a single-family, gas-heated house located in Nordwijk, NL.
Messages generated by (approx. 30) smart home devices have been daily collected over 32 months.
The devices include: power/gas meters, wall electricity sockets, wall switches, light bulbs, temperature/humidity sensors, motion sensors, door sensors, etc.
On average, the devices generate daily more than 2000 messages about their state (e.g. *current temperature is 20.1°C*, *the light has been turned on*, *the door has been opened*, *motion has been detected*, etc.)

The messages are provided by several sources listed below.

### Source `SmartThings`

The [SmartThings](https://www.smartthings.com/) platform collect messages from many smart home devices:

- The messages are provided as multiple files in TSV format.
- File names resemble: `smartthings/smartthings.202301.tsv.gz`.

Columns:

- `loc` - physical location of the device (e.g. `kitchen`, `living room`, `bathroom`, `garden`)
- `level` - house floor where the device is located (e.g. `ground`)
- `name` - name of the device, given by the owner
- `epoch` - message timestamp (a number, search for *Unix Time Stamp*)
- `capability` - what the message describes (e.g. `temperatureMeasurement`)
- `attribute` - measured property (e.g. `temperature`)
- `value` and `unit` provide the device state readings (e.g. `20.1` and `°C`)

Example:

```
loc     level   name                 epoch           capability         attribute value  unit
kitchen ground  Kitchen (table)      1665386804      switch               switch  off     
living  ground  Living Room (sound)  1665412626      switch               switch  on      
living  ground  Living Room (TV)     1665412631      switch               switch  on      
kitchen ground  Door (main)          1665415150      voltageMeasurement   voltage 3.035   V
living  ground  Living Room (TV)     1665415330      switch               switch  off     
living  ground  Living Room (sound)  1665415330      switch               switch  off     
kitchen ground  Door (main)          1665418160      voltageMeasurement   voltage 3.025   V
kitchen ground  Door (main)          1665418160      signalStrength       lqi     220     
```

### Source `P1e`

`P1e` source provides messages about electricity usage:

- Multiple files with readings from a smart meter in compressed CSV format.
- Collected approx. biweekly.
- 15-minutes resolution.
- File names start with `P1e-`.
- Rows with identical readings might be present in multiple files.
- The files are compressed with `gzip` (check the `open` function in `gzip` module).

Columns:

- Reading time (in Europe/Amsterdam timezone)
- Total energy usage [kWh] in low-cost hours (`T1`)
- Total energy usage [kWh] in high-cost hours (`T2`)
- (ignore other columns)

Example (a fragment of a `P1e-` file):

```
time,Electricity imported T1,Electricity imported T2,Electricity exported T1,Electricity exported T2
2022-12-01 00:00,7937.977,6284.179,0.000,0.000
2022-12-01 00:15,7938.024,6284.179,0.000,0.000
2022-12-01 00:30,7938.075,6284.179,0.000,0.000
2022-12-01 00:45,7938.178,6284.179,0.000,0.000
```

### Source `P1g`

`P1g` source provides messages about gas usage:

- Multiple files with readings from a smart meter.
- Collected approx. biweekly.
- 15-minutes resolution.
- File names start with `P1g-`.
- Rows with identical readings might be present in multiple files.
- The files are compressed with `gzip` (check the `open` function in `gzip` module).

Columns:
- Reading time (as text, in Europe/Amsterdam timezone)
- Total gas usage [m3].

### Source `OpenWeatherMap`

Study [the Historical Weather page of Open-meteo](https://open-meteo.com/en/docs/historical-weather-api). Using a link generated on that page, you can download the weather data for a specific location and time period. The temperature and humidity data are available with 1 hour resolution. The data is provided in JSON format.
Look into the course materials (dict/set exercises) for examples of how to access the data.

## Database design

Data from the sources are not uniform. They are provided in different formats (TSV, CSV, JSON) and they may contain duplicated entries. The goal of the project is to remove duplicates and store all the data in a relational database (SQLite).

You are free to design the database schema as you wish.

Date/time columns should be stored in the database as an integer number (called `epoch`) representing the [Unix Time](https://en.wikipedia.org/wiki/Unix_time) (number of seconds since 1970-01-01 00:00:00 UTC). Libraries allowing to convert between different date/time formats and epoch time are available in Python.

## Database access module

The database should be accessed through a class interface:

- You need to implement a class called `HomeMessagesDB`.
- The class should be implemented in a separate file called `home_messages_db.py`.
- An instance of the class should be initialized with the database file name (in SQLAlchemy format, e.g. `sqlite:///myhome.db`). The database file should be created if it does not exist.
- The class should provide methods to:
    - Create the database (if it does not exist).
    - Insert data into the database (as needed by the command-line tools, see below).
    - Query the database (as needed by the reports, see below).

**Note:** Any direct access to the database must be done in a method of the class. No SQL code can appear outside of this class.

## Command-line tools to insert data into the database

The task is to write command-line tools inserting data into the database:

- For each data source there should be a separate tool (named: `smartthings.py`, `p1e.py`, `p1g.py`, `openweathermap.py`).
- The tools should be Python scripts (`.py`) which can be executed from the terminal/shell command line.
- Two students should be responsible for each tool and each student should contribute to at least one tool. The contributions should be visible in the git history.
- To get access to the database the tool should `import home_messages_db` module, create an instance of the `HomeMessagesDB` class and communicate with the database through the methods of the class. No direct access to the database is allowed in the tools.
- When used without any arguments, or with `-h` or `--help` option, the tool should print a help message describing the tool and its options (see the [Click](https://click.palletsprojects.com/en/7.x/) library).
- When used with wrong arguments, the tool should raise an exception which would print an error message.
- For top grades additional options can be added to the tools (for example: show number of entries already present in the database, automatically handle compressed files, etc.). These additional options must be clearly documented in the `README.md`.

### SmartThings tool

This tool should insert data from the SmartThings files into the database.  

Here is the description of the arguments and options of the tool (this should be printed when the tool is called with `-h` or `--help` option):
```
Usage:
    smartthings.py [OPTIONS] smartthingsLog.1.tsv [smartthingsLog.2.tsv...]

Output options:
    -d DBURL insert into the project database (DBURL is a SQLAlchemy database URL)
```

For example, to load the data from the file `smartthings/smartthings.202301.tsv` into the database `myhome.db` use the command:

```bash
smartthings.py -d sqlite:///myhome.db smartthings/smartthings.202301.tsv
```

And to load the data from all the files starting with `smartthings/smartthings.` into the database `myhome.db` use the command:

```bash
smartthings.py -d sqlite:///myhome.db smartthings/smartthings.*
```

### P1e tool

This tool should insert electricity consumption data from the P1e files into the database.

Here is the description of the arguments and options of the tool (this should be printed when the tool is called with `-h` or `--help` option):
```
Usage:
    p1e.py [OPTIONS] P1e-2022-12-01-2023-01-10.csv.gz [...]

Output options:
    -d DBURL insert into the project database (DBURL is a SQLAlchemy database URL)
```

For example, to load the data from the file `P1e-2022-12-01-2023-01-10.csv.gz` into the database `myhome.db` use the command:

```bash
p1e.py -d sqlite:///myhome.db P1e-2022-12-01-2023-01-10.csv.gz
```

And to load the data from all the files matching the filename pattern `P1e-*.csv.gz` into the database `myhome.db` use the command:

```bash
p1e.py -d sqlite:///myhome.db P1e-*.csv.gz
```

### P1g tool

This tool should insert gas consumption data from the P1g files into the database. It should follow the same design as the P1e tool.

### OpenWeatherMap tool

Weather data can be obtained freely from the web site mentioned above. The weather data must be used in at least one report but it does not need to be stored in the database (it can be downloaded from the website when needed). The group should decide whether to store the data in the database or not. If the data is stored in the database, then there should be a command line tool which follows a similar design as the other tools.

**Note:** The access to the weather database is free. You are not expected to pay for the access to the weather data. Please contact the teacher in case of any problems with the access to the weather data.

### Database tool

The group may decide to create a separate tool which would allow to initialize the database (create the tables). It should be described in the `README.md` file how to initialize the database and how to use all the tools to fill the database with the data from the source files.

## Data analyses

The task is to write several Python notebooks reports (`.ipynb`) with analyses of the data from the database:

- Each report should discuss at least one question about the data, contain at least one visualization and at least one table (e.g. a table with statistics or a table with the results of a query)
- Two students should be responsible for each report and each student should contribute to at least one notebook report. The contributions should be visible in the git history.
- The names of the report files should start with `report_` (e.g. `report_gas_usage.ipynb`).
- To get access to the data the report should `import home_messages_db`, create an instance of the `HomeMessagesDB` class and communicate with the database through the methods of the class. The report is not allowed to access the database directly.
- At least one of the reports should use weather data obtained from the weather server.

Here are some example questions which can be studied in a report (you're welcome to create your own questions):

- How to identify time intervals when nobody is at home?
- What is the distribution of the energy and gas usage over a day?
- Are there weekly patterns in the energy and gas usage?
- When heating is off, how quickly does the temperature drop? Does this depend on the outside temperature?
- How long per day are the lights in the living room on? Does it depend on the length of the day?
- The devices are not ideal - how to identify intervals when a device is not working?
- What is the difference between the measured (garden) and predicted (from the weather server; for Nordwijk) temperature?

For top grades a report needs to contain an additional statistical component (e.g. a regression, a clustering, a hypothesis test).

## Submission

The project should be created inside a new dedicated, **private** git repository. **The name of the repository must start with `GA2025_xx` (where `xx` should be replaced with the group number).** The repository should contain at least the following files:

- `README.md` file with the description of the project, project files, and instructions how to run the tools and the reports. 
- `home_messages_db.py` file with a documented implementation of the `HomeMessagesDB` class (and eventually other classes).
- The command line tools (Python scripts).
- The reports (Python notebooks).

In the `README.md` file there should be a section `Contributions` containing a **manually written table describing who is the author of each project file**. The table should contain one row for each student and the columns should be: `FullName` (of the student), `StudentId` (the `s123...` number of the student), `GitHubName` (the username of the student in GitHub), `Files` (comma-separated names of the files which the student contributed to). All files not listed in the table will be considered as authored by everybody in the group.

The GitHub group repository should be shared with the same teachers as the assignment repositories. Before sharing, double check that the name of the repository starts with `GA2025_`.

## Grading

The primarily goal of the project is to learn how to work in a group on a software project. Therefore, the central criteria for grading the project is whether the designed class, the command line tools and the reports are consistent with each other.

Moreover, the project will be graded based on the following criteria:

- The `README.md` file is well written and contains all the necessary information about the project (and also about who is responsible for which report and command line tools).
- The SQLite database is present and is used to store the data. No duplicate data are stored in the database.
- The command line tools work as expected, are well documented and easy to use. Error messages are generated and informative.
- The reports are well written, easy to understand and contain interesting analyses (with a plot and a table).
- The project is delivered on time.
- Time/date is handled well (e.g. the time zone is taken into account). Temperatures/humidity are in valid ranges.
- The commit history confirms contributions of declared students.

**Note:** By default all students in the group will get the same grade.
However, in case of a significant difference in the contributions of the students, individual grading may be requested. In that case please state the wish in the `README.md` file. Then, each file of the project will be graded separately. The final grade for each student will be the average of the grades for the individual files to which the student contributed.
